# Session 6 · Graph Analysis for Fintech Fraud Detection
## BestAgro Lab — Kreo Digital

**One integrated lab:** data → graph → supervised model → anomaly detection → hybrid risk → analyst review.

**Main dataset:** Kaggle IEEE-CIS. **Database:** DuckDB, with an optional DuckPGQ exercise. No Neo4j.

Allow 4–5 hours. Work through cells in order. Questions are embedded after each stage; add your answers in the supplied response cells. Runnable cells provide reference implementations so you can experiment and compare outcomes.

### Learning outcomes
- Construct a transaction–attribute incidence graph without inventing customer identities.
- Explore degree, shared relationships, multi-hop paths and time-ordered motifs.
- Compute historical graph features without observing the current or future transactions.
- Compare supervised, unsupervised and graph-enriched methods.
- Keep one hybrid risk score and separate its review policy.
- Evaluate graph contribution using later data and a fixed review budget.

### Important interpretation
IEEE-CIS does not provide a complete account-to-account transfer network. Card attributes, device descriptions and email domains are **proxies**, not unique people, cards or hardware IDs. Transfer fan-in/out and circular funds-flow are demonstrated in a small, explicitly synthetic teaching example inside this same notebook; its labels and results are never mixed with IEEE-CIS.


## 1. Environment and Kaggle data
Accept the [competition rules](https://www.kaggle.com/competitions/ieee-fraud-detection/rules) yourself, then download `train_transaction.csv` and `train_identity.csv` from the [data page](https://www.kaggle.com/competitions/ieee-fraud-detection/data). Extract any ZIPs into `data/raw/ieee_cis` beside this notebook, or change `DATA_DIR` below.

The rules specify non-commercial educational/research use. Keep raw competition data, credentials and row-level exports out of GitHub. Official Kaggle test labels are not public; our test is a later slice of the labelled training files.

Install dependencies in your selected notebook kernel using the next cell once. After installation, restart the kernel if requested. DuckDB is pinned to 1.4.4 for the optional DuckPGQ compatibility described by its current documentation.


In [ ]:
# Uncomment to install in the current Jupyter kernel, then restart if necessary.
# %pip install duckdb==1.4.4 numpy==2.3.5 scikit-learn==1.8.0 matplotlib==3.10.8


In [ ]:
from pathlib import Path
from types import SimpleNamespace
import os, json
import duckdb
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Set the environment override only for automated fixture testing.
DATA_DIR = Path(os.environ.get('SESSION6_DATA_DIR', 'data/raw/ieee_cis'))
OUTPUT_DIR = Path(os.environ.get('SESSION6_OUTPUT_DIR', 'outputs/session06'))
LIMIT = int(os.environ.get('SESSION6_LIMIT', '60000'))  # 0 = full labelled training data
EXPORT_ROW_LEVEL = False  # enable only for local investigation; never commit raw-derived outputs
RUN_DUCKPGQ = False  # optional network-dependent exercise
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
a = SimpleNamespace(data_dir=DATA_DIR, output_dir=OUTPUT_DIR, limit=LIMIT)
for filename in ['train_transaction.csv', 'train_identity.csv']:
    if not (DATA_DIR / filename).is_file():
        raise FileNotFoundError(f'Download and extract {filename} into {DATA_DIR.resolve()} first')
c = duckdb.connect()
print('DuckDB', duckdb.__version__, '| Data folder:', DATA_DIR.resolve())


## 2. Load and audit the labelled data
We join on `TransactionID` using a **left join**, so missing identity data does not delete a transaction. `TransactionDT` is a relative time offset, not a real calendar date.

The row limit chooses the earliest chronological observations. Input CSVs are initially loaded in full, so this limit does not bound ingestion memory. A short prefix may have different prevalence from the full dataset.


In [ ]:
# Parameterized input paths; read IDs as strings to preserve anonymized categories.
c.execute('CREATE TABLE tx AS SELECT * FROM read_csv_auto(?,all_varchar=true)',[str(a.data_dir/'train_transaction.csv')])
c.execute('CREATE TABLE ident AS SELECT * FROM read_csv_auto(?,all_varchar=true)',[str(a.data_dir/'train_identity.csv')])
for table in ['tx','ident']:
    n,u=c.execute(f'SELECT count(*),count(DISTINCT TransactionID) FROM {table}').fetchone()
    if n!=u: raise ValueError(f'Duplicate or null TransactionID in {table}')
required=['TransactionID','TransactionDT','TransactionAmt','ProductCD','card1','card2','card3','card4','card5','card6','addr1','addr2','P_emaildomain','isFraud']
if not set(required)<=set(r[0] for r in c.execute('DESCRIBE tx').fetchall()):
    raise ValueError('Missing required transaction columns')
c.execute('''CREATE TABLE joined AS SELECT t.*,i.DeviceInfo,i.DeviceType
FROM tx t LEFT JOIN ident i USING(TransactionID)''')
c.execute('''CREATE TABLE ordered AS SELECT *,CAST(TransactionDT AS BIGINT) AS dt,
CAST(TransactionAmt AS DOUBLE) AS amount,CAST(isFraud AS INTEGER) AS y
FROM joined ORDER BY CAST(TransactionDT AS BIGINT),TransactionID''')
if a.limit:
    if a.limit<100: raise ValueError('Use at least 100 rows')
    c.execute('DELETE FROM ordered WHERE TransactionID NOT IN (SELECT TransactionID FROM ordered ORDER BY dt,TransactionID LIMIT ?)',[a.limit])


In [ ]:
assert c.execute('SELECT count(*) FROM joined').fetchone()[0] == c.execute('SELECT count(*) FROM tx').fetchone()[0]
print('Transaction rows:', c.execute('SELECT count(*) FROM tx').fetchone()[0])
print('Selected rows:', c.execute('SELECT count(*) FROM ordered').fetchone()[0])
print('Selected fraud prevalence:', c.execute('SELECT avg(y) FROM ordered').fetchone()[0])
print('Missing device-description rate:', c.execute('SELECT avg(CASE WHEN DeviceInfo IS NULL THEN 1.0 ELSE 0.0 END) FROM ordered').fetchone()[0])


### Questions 1–3
1. Why would an inner join bias this experiment?
2. Why can we not call the offset a real date or infer Jakarta local hour from it?
3. What can change when using only the earliest 60,000 transactions?


**Your responses (1–3):** Write your reasoning here.

## 3. Build the graph and temporal split
The graph has transaction vertices and typed attribute vertices. An edge means “this transaction has this attribute value”. Missing values create **no edge**. A structured card1–card6 tuple is an experimental proxy, not a verified card identity.

Distinct time values define approximately 60/20/20 train/calibration/test partitions. Tied timestamps remain together. Shared proxies may recur across splits intentionally: this is known-entity online replay, not a cold-start benchmark.


In [ ]:
# Fixed time boundaries, never divide transactions with the same timestamp.
times=[r[0] for r in c.execute('SELECT DISTINCT dt FROM ordered ORDER BY dt').fetchall()]
if len(times)<10: raise ValueError('Insufficient distinct times')
b1=times[int(len(times)*.60)];b2=times[int(len(times)*.80)]
c.execute('''CREATE TABLE nodes AS SELECT *,
CASE WHEN card1 IS NOT NULL AND card2 IS NOT NULL AND card3 IS NOT NULL
AND card4 IS NOT NULL AND card5 IS NOT NULL AND card6 IS NOT NULL
THEN CAST(to_json(list_value(card1,card2,card3,card4,card5,card6)) AS VARCHAR) END AS card_proxy
FROM ordered''')
# Incidence graph: transaction -> attribute value. Missing values create NO edge.
# DeviceInfo, domains and card composites are proxies, NOT unique real entities.
c.execute('''CREATE TABLE edges AS
SELECT TransactionID,dt,'card_proxy' AS kind,card_proxy AS value FROM nodes WHERE card_proxy IS NOT NULL
UNION ALL SELECT TransactionID,dt,'device_attribute',DeviceInfo FROM nodes WHERE DeviceInfo IS NOT NULL
UNION ALL SELECT TransactionID,dt,'email_domain',P_emaildomain FROM nodes WHERE P_emaildomain IS NOT NULL''')


In [ ]:
print('Edges by attribute kind:', c.execute('SELECT kind,count(*),count(DISTINCT value) FROM edges GROUP BY kind ORDER BY kind').fetchall())
print('Most connected attribute hubs:', c.execute('SELECT kind,value,count(*) AS degree FROM edges GROUP BY kind,value ORDER BY degree DESC LIMIT 8').fetchall())
assert c.execute('SELECT count(*) FROM edges WHERE value IS NULL').fetchone()[0] == 0


## 4. Explore an actual transaction neighborhood
Select a small transaction neighborhood and visualize its attribute links. This picture is a local explanatory view, not a fraud verdict. We cap the neighborhood at 12 transactions so a popular domain does not create an unreadable plot.


In [ ]:
seed = c.execute('SELECT TransactionID FROM edges ORDER BY dt,TransactionID LIMIT 1').fetchone()[0]
neighbor_ids = [r[0] for r in c.execute("""
SELECT DISTINCT e.TransactionID FROM edges e JOIN edges s
ON e.kind=s.kind AND e.value=s.value WHERE s.TransactionID=?
ORDER BY e.TransactionID LIMIT 12
""", [seed]).fetchall()]
small_edges = c.execute('SELECT TransactionID,kind,value FROM edges WHERE TransactionID IN (SELECT unnest(?))', [neighbor_ids]).fetchall()
attrs = sorted(set((kind,value) for _,kind,value in small_edges))
pos_tx = {tid:(0,-j) for j,tid in enumerate(neighbor_ids)}
pos_attr = {key:(1,-j*max(len(neighbor_ids)-1,1)/max(len(attrs)-1,1)) for j,key in enumerate(attrs)}
fig, ax = plt.subplots(figsize=(12,7))
for tid,kind,value in small_edges:
    x1,y1=pos_tx[tid]; x2,y2=pos_attr[(kind,value)]
    ax.plot([x1,x2],[y1,y2],color='#c7d2de',linewidth=.8,zorder=1)
for tid,(x,ypos) in pos_tx.items():
    ax.scatter(x,ypos,color='#176b87',s=80,zorder=2)
    ax.text(x-.025,ypos,'Tx '+tid,ha='right',va='center',fontsize=8)
for (kind,value),(x,ypos) in pos_attr.items():
    ax.scatter(x,ypos,color='#e99a35',s=80,zorder=2)
    ax.text(x+.025,ypos,kind+': '+value[:42],va='center',fontsize=8)
ax.set_xlim(-.3,2);ax.set_title('Local transaction–attribute graph; identifiers are dataset proxies')
ax.axis('off');plt.tight_layout();plt.show()


### Shared attributes and bounded multi-hop paths
A transaction → attribute ← transaction path means shared attributes. Two different attribute types can connect a longer chain; it does not prove shared ownership. The next query filters the historical graph to a fixed cutoff and uses only card/device proxies, avoiding broad email-domain hubs. A bounded seed/time range helps control join expansion.


In [ ]:
cutoff = int(b1)
c.execute("""CREATE OR REPLACE TEMP TABLE neighborhood_edges AS
SELECT TransactionID,kind,value FROM edges
WHERE dt < ? AND dt >= ? AND kind IN ('card_proxy','device_attribute')""", [cutoff, cutoff-604800])
seed2 = c.execute('SELECT TransactionID FROM neighborhood_edges ORDER BY TransactionID LIMIT 1').fetchone()
paths = [] if seed2 is None else c.execute("""
WITH one_hop AS (
 SELECT DISTINCT b.TransactionID AS middle,a.kind AS first_link
 FROM neighborhood_edges a JOIN neighborhood_edges b
 ON a.kind=b.kind AND a.value=b.value
 WHERE a.TransactionID=? AND a.TransactionID<>b.TransactionID
 ORDER BY middle LIMIT 100
)
SELECT DISTINCT h.middle,d.TransactionID AS destination,h.first_link,c.kind AS second_link
FROM one_hop h JOIN neighborhood_edges c ON h.middle=c.TransactionID
JOIN neighborhood_edges d ON c.kind=d.kind AND c.value=d.value
WHERE d.TransactionID<>h.middle AND d.TransactionID<>? AND c.kind<>h.first_link
LIMIT 20
""", [seed2[0],seed2[0]]).fetchall()
print('Paths with two shared-attribute links (up to 20):', paths)


### Questions 4–7
4. Why is `DeviceInfo` unsuitable as a unique device fingerprint?
5. What goes wrong if every missing value becomes a common “unknown” vertex?
6. Compare card-proxy, device-description and email-domain hub sizes. Which likely contain benign connections?
7. Does the multi-hop query show temporal money flow? Explain what each hop actually means.


**Your responses (4–7):** Write your reasoning here.

## 5. Understand transfer motifs with one small illustration
This synthetic illustration teaches the transfer vocabulary from the presentation. **It is not IEEE-CIS and is not model training data.** Collection and dispersal may also be legitimate business activity. Even an ordered circular sequence does not prove laundering or conservation of the same money.


In [ ]:
toy = duckdb.connect()
toy.execute('CREATE TABLE payments(src VARCHAR,dst VARCHAR,minute INTEGER,amount DOUBLE)')
toy.executemany('INSERT INTO payments VALUES (?,?,?,?)', [
 ('A','H',1,100),('B','H',2,100),('C','H',3,100),
 ('H','D',5,80),('H','E',6,80),('D','A',8,75),('A','H',9,70)])
print('Fan-in:', toy.execute('SELECT dst,count(DISTINCT src) FROM payments GROUP BY dst ORDER BY 2 DESC').fetchall())
print('Fan-out:', toy.execute('SELECT src,count(DISTINCT dst) FROM payments GROUP BY src ORDER BY 2 DESC').fetchall())
print('Ordered cycles:', toy.execute("""SELECT a.src,a.dst,b.dst,a.minute,b.minute,c.minute
FROM payments a JOIN payments b ON a.dst=b.src JOIN payments c ON b.dst=c.src AND c.dst=a.src
WHERE a.src<>a.dst AND a.src<>b.dst AND a.dst<>b.dst
AND a.minute<b.minute AND b.minute<c.minute AND c.minute-a.minute<=30""").fetchall())
print('Bounded reachability:', toy.execute("""WITH RECURSIVE paths AS (
 SELECT src AS origin,dst AS node,1 AS depth,[src,dst] AS visited FROM payments WHERE src<>dst
 UNION ALL SELECT p.origin,t.dst,p.depth+1,list_append(p.visited,t.dst)
 FROM paths p JOIN payments t ON p.node=t.src
 WHERE p.depth<3 AND NOT list_contains(p.visited,t.dst))
 SELECT origin,node,min(depth) FROM paths GROUP BY origin,node ORDER BY origin,node""").fetchall())
toy.close()


### Questions 8–9
8. Which node collects from several distinct senders? Why does repeated A → H not add another distinct sender?
9. Why must these transfer motifs not be presented as actual IEEE-CIS account-transfer findings?


**Your responses (8–9):** Write your reasoning here.

## 6. Point-in-time graph features
Compute prior seven-day transaction counts for each linked attribute. The window ends at `dt - 1`: current and same-time events are excluded. Earlier unlabelled calibration/test observations can enter history for later observations. All labels are excluded from graph features.

Actual ingestion and label-confirmation times are not supplied here. We assume observation availability at transaction time and label availability at model-fitting time; realistic delays require a separate sensitivity analysis. Do not describe this as fully verified production point-in-time replay.


In [ ]:
# RANGE ends at dt-1; tied-time events cannot observe each other. 7-day lookback.
c.execute('''CREATE TABLE history AS SELECT *,
count(*) OVER(PARTITION BY kind,value ORDER BY dt RANGE BETWEEN 604800 PRECEDING AND 1 PRECEDING) AS prior_7d
FROM edges''')
c.execute('''CREATE TABLE features AS SELECT n.TransactionID,n.dt,n.y,n.amount,
CASE WHEN n.DeviceInfo IS NULL THEN 1 ELSE 0 END AS device_missing,
CASE WHEN n.card_proxy IS NULL THEN 1 ELSE 0 END AS card_proxy_missing,
coalesce(max(CASE WHEN h.kind='card_proxy' THEN h.prior_7d END),0) AS card_history,
coalesce(max(CASE WHEN h.kind='device_attribute' THEN h.prior_7d END),0) AS device_history,
coalesce(max(CASE WHEN h.kind='email_domain' THEN h.prior_7d END),0) AS domain_history
FROM nodes n LEFT JOIN history h USING(TransactionID)
GROUP BY n.TransactionID,n.dt,n.y,n.amount,n.DeviceInfo,n.card_proxy''')
rows=c.execute('SELECT * FROM features ORDER BY dt,TransactionID').fetchall()
y=np.array([r[2] for r in rows]); dt=np.array([r[1] for r in rows])
X=np.array([[float(v) if v is not None else np.nan for v in r[3:]] for r in rows])
if not np.all(np.isin(y,[0,1])): raise ValueError('isFraud must be binary')
if not np.isfinite(X[:,0]).all(): raise ValueError('Invalid transaction amounts')
train=dt<b1;cal=(dt>=b1)&(dt<b2);test=dt>=b2
for name,mask in [('train',train),('calibration',cal),('test',test)]:
    if len(np.unique(y[mask]))!=2: raise ValueError(f'{name} needs both classes; increase --limit')


In [ ]:
# Independent semantic check: tied times cannot observe each other.
assert c.execute("""SELECT count(*) FROM history h WHERE h.prior_7d<>(
 SELECT count(*) FROM edges e WHERE e.kind=h.kind AND e.value=h.value
 AND e.dt>=h.dt-604800 AND e.dt<h.dt)
AND h.TransactionID IN (SELECT TransactionID FROM ordered ORDER BY dt,TransactionID LIMIT 20)
""").fetchone()[0] == 0
print('Feature order: amount, device_missing, card_proxy_missing, card_history, device_history, domain_history')
print('Train/calibration/test:', int(train.sum()),int(cal.sum()),int(test.sum()))


### Questions 10–12
10. Why does the graph SQL end one second before the current offset?
11. What must change if observations arrive a day late or labels take 30 days to confirm?
12. Propose a cold-start evaluation distinct from this known-entity temporal replay.


**Your responses (10–12):** Write your reasoning here.

## 7. Supervised modelling
Fit a modest histogram gradient-boosting baseline with amount and missingness indicators. Then add the three historical graph degrees. These are local graph features, not a GNN or community detector. The baseline is deliberately small and not an exhaustive IEEE-CIS benchmark.


In [ ]:
# Modest numeric baseline for teaching; not a leaderboard-complete feature set.
# No current/future labels or historical fraud rates are graph inputs.
prob={}
for name,idx in [('transaction',list(range(3))),('graph',list(range(6)))]:
    m=HistGradientBoostingClassifier(max_iter=80,max_leaf_nodes=15,random_state=42)
    m.fit(X[train][:,idx],y[train]);prob[name]=m.predict_proba(X[:,idx])[:,1]


## 8. Unsupervised anomaly detection and rules
Isolation Forest fits the unlabelled mixed training population. Map anomaly scores to training-referenced empirical ranks. These ranks are **not fraud probabilities**. A simple amount rule uses the training 95th percentile. A topology activity rank summarizes graph history; high activity is not intrinsically malicious.


In [ ]:
iso=IsolationForest(n_estimators=100,random_state=42,n_jobs=1)
iso.fit(X[train,:3]);raw=-iso.score_samples(X[:,:3])
ref=np.sort(raw[train]);anomaly=np.searchsorted(ref,raw,side='right')/len(ref)
# Training-only reference for an illustrative high-amount rule.
threshold=np.quantile(X[train,0],.95)
rule=(X[:,0]>threshold).astype(float)
# Graph activity is a structural frequency signal, not a fraud probability.
gref=np.sort(np.log1p(X[train,3:].sum(axis=1)))
graw=np.log1p(X[:,3:].sum(axis=1))
gscore=np.searchsorted(gref,graw,side='right')/len(gref)


### Questions 13–15
13. Which features enter the transaction baseline versus graph-enriched model?
14. Can a legitimate transaction be anomalous? Give an operational example.
15. Why are empirical anomaly ranks different from calibrated probabilities?


**Your responses (13–15):** Write your reasoning here.

## 9. Hybrid risk scoring — the single decision score
First reproduce the presentation example on a 0–100 scale:

`Risk = 0.15 × rules + 0.40 × supervised + 0.15 × anomaly + 0.30 × graph`

These weights are illustrative, not optimized production settings. Next learn a logistic combination using only the later calibration period. Compare a hybrid without graph signals with one using graph-enriched predictions and graph activity. Probability calibration must be assessed, not assumed.


In [ ]:
assert abs(.15*30+.40*45+.15*70+.30*95-61.5)<1e-10
fixed_hybrid = .15*rule + .40*prob['transaction'] + .15*anomaly + .30*gscore
print('Deck worked example: 61.5 / 100')


In [ ]:
scores={'supervised_transaction':prob['transaction'],'supervised_graph':prob['graph']}
for name,z in [('hybrid_without_graph',np.column_stack([rule,prob['transaction'],anomaly])),
               ('hybrid_with_graph',np.column_stack([rule,prob['graph'],anomaly,gscore]))]:
    m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=42))
    m.fit(z[cal],y[cal]);scores[name]=m.predict_proba(z)[:,1]


In [ ]:
scores['fixed_weight_hybrid'] = fixed_hybrid


### Questions 16–18
16. Explain each of the four signals in the hybrid score.
17. Why should the fixed weighted score not be labelled a calibrated fraud probability?
18. Why fit combination weights on calibration data and reserve the final test period?


**Your responses (16–18):** Write your reasoning here.

## 10. Evaluate all models on the same later transactions
Report average precision, precision@K, recall@K and prevalence together. K is a global budget of up to 100 reviews across the test period, not 100 per day. Brier score is reported only for model probability estimates; the fixed heuristic risk remains a ranking score.


In [ ]:
report={'rows':len(rows),'split_counts':{'train':int(train.sum()),'calibration':int(cal.sum()),'test':int(test.sum())},
        'time_boundaries':[int(b1),int(b2)],'label_availability_assumption':'Training labels available at fitting; actual confirmation times unavailable',
        'protocol':'Online replay: later transactions may use earlier unlabelled observations, never same-time or future observations',
        'metrics':{}}
for name,s in scores.items():
    yy=y[test];ss=s[test];k=min(100,len(yy));top=np.argsort(-ss,kind='stable')[:k]
    report['metrics'][name]={'average_precision':float(average_precision_score(yy,ss)),
    'brier_score':float(brier_score_loss(yy,ss)),'precision_at_k':float(yy[top].mean()),
    'recall_at_k':float(yy[top].sum()/yy.sum()),'k':k,'prevalence':float(yy.mean())}
(a.output_dir/'metrics.json').write_text(json.dumps(report,indent=2)+'\n')


In [ ]:
report['metrics']['fixed_weight_hybrid'].pop('brier_score', None)
(a.output_dir/'metrics.json').write_text(json.dumps(report,indent=2)+'\n')
for name, values in report['metrics'].items():
    print(name, values)
fig,axes=plt.subplots(1,2,figsize=(12,4))
for name,score in scores.items():
    precision,recall,_=precision_recall_curve(y[test],score[test])
    axes[0].plot(recall,precision,label=name)
axes[0].axhline(y[test].mean(),color='gray',linestyle='--',label='prevalence')
axes[0].set(xlabel='Recall',ylabel='Precision',title='Later-period precision–recall')
axes[0].legend(fontsize=7)
names=list(report['metrics']);aps=[report['metrics'][name]['average_precision'] for name in names]
axes[1].barh(names,aps,color='#176b87');axes[1].set(xlabel='Average precision',title='Same holdout, different signals')
plt.tight_layout();plt.show()
if EXPORT_ROW_LEVEL:
    for table in ['features','edges']:
        dest=str(a.output_dir/f'{table}.parquet').replace("'","''")
        c.execute(f"COPY {table} TO '{dest}' (FORMAT PARQUET)")


### Questions 19–22
19. Did graph features improve average precision and precision@K in your run? Report the actual values, including any regression.
20. Why is this not a Kaggle leaderboard score or evidence of fraud losses prevented?
21. Remove each graph feature family and rerun the model/ensemble. Which contributes most? Keep the same test window and budget.
22. How could train-only normalization still fail under drift? Inspect probability bins and Brier score for miscalibration.


**Your responses (19–22):** Include your experiment table and interpretation here.

## 11. Analyst review and decision policy
Risk estimates and actions are separate. Select a review threshold on calibration data only. The toy policy targets about 10% review volume there; actual holdout volume may differ. Missing device information is visible in the features and baseline. Missing graph relationships yield zero history and proxy-missing indicators; this is not the same as an unavailable graph service. Production needs explicit service-health/freshness fallback policies.

The score contributions shown below are signal values, not SHAP explanations or evidence of fraud. This lab never automatically blocks payments.


In [ ]:
final=scores['hybrid_with_graph']
review_threshold=float(np.quantile(final[cal],.90))
indices=np.flatnonzero(test & (final>=review_threshold))
indices=sorted(indices,key=lambda j:(-final[j],rows[j][0]))
print('Calibration-selected review threshold:',review_threshold)
print('Holdout transactions queued:',len(indices),'of',int(test.sum()))
for j in indices[:10]:
    print({'TransactionID':rows[j][0],'hybrid_risk_0_100':round(float(final[j])*100,2),
           'rule':float(rule[j]),'supervised_graph':round(float(prob['graph'][j]),3),
           'anomaly':round(float(anomaly[j]),3),'graph_activity':round(float(gscore[j]),3),
           'action':'manual_review'})


### Questions 23–25
23. Is the threshold queue identical to top-100 evaluation? Explain the difference.
24. For one queued transaction, list evidence, benign alternatives and what additional evidence an analyst needs.
25. Define fallback behavior when graph features are stale or unavailable. What would you monitor after deployment?


**Your responses (23–25):** Write your investigation note and policy here.

## 12. Optional native graph query with DuckPGQ
This section belongs to the same lab and reads its existing historical edges. It is optional because extension downloads may be unavailable. The official guide currently recommends DuckDB 1.4.4 for DuckPGQ and warns of incomplete features. These statements are documentation-based and not execution-verified in the authoring environment. The required lab above uses DuckDB SQL.

Set `RUN_DUCKPGQ=True` in configuration to attempt the extension. The graph is a pre-cutoff snapshot. Attribute keys encode type and value together. The query shows a shared-attribute pattern, not temporal flow.


In [ ]:
if RUN_DUCKPGQ:
    c.execute('INSTALL duckpgq FROM community')
    c.execute('LOAD duckpgq')
    c.execute("""CREATE OR REPLACE TABLE graph_links AS
    SELECT TransactionID,CAST(to_json(list_value(kind,value)) AS VARCHAR) AS attribute_id
    FROM edges WHERE dt < ?""",[int(b1)])
    c.execute('CREATE OR REPLACE TABLE graph_transactions AS SELECT DISTINCT TransactionID FROM graph_links')
    c.execute('CREATE OR REPLACE TABLE graph_attributes AS SELECT DISTINCT attribute_id FROM graph_links')
    c.execute("""CREATE PROPERTY GRAPH ieee_graph
    VERTEX TABLES (graph_transactions,graph_attributes)
    EDGE TABLES (graph_links SOURCE KEY (TransactionID) REFERENCES graph_transactions (TransactionID)
    DESTINATION KEY (attribute_id) REFERENCES graph_attributes (attribute_id))""")
    print(c.execute("""SELECT * FROM GRAPH_TABLE (ieee_graph
    MATCH (a:graph_transactions)-[x:graph_links]->(v:graph_attributes)<-[z:graph_links]-(b:graph_transactions)
    WHERE a.TransactionID < b.TransactionID
    COLUMNS (a.TransactionID AS first_transaction,b.TransactionID AS second_transaction,v.attribute_id AS shared_attribute)
    ) LIMIT 10""").fetchall())
    c.execute('DROP PROPERTY GRAPH ieee_graph')
else:
    print('Optional DuckPGQ section skipped; all required graph analysis used DuckDB SQL.')


## Submission checklist and marking
Submit this notebook with written answers, local executed outputs, metric comparison and an analyst note. Do not publish raw/row-level Kaggle-derived outputs without confirming the data terms. Clear outputs before committing to the public repository.

| Area | Points | Evidence |
|---|---:|---|
| Data and graph interpretation | 20 | Correct join, proxy limitations and missing-value handling |
| Graph exploration | 20 | Neighborhood, multi-hop explanation and motif distinction |
| Temporal correctness | 20 | Strict historical windows; label/arrival assumptions |
| Models and hybrid scoring | 25 | Matched-budget comparison and ablations |
| Investigation and policy | 15 | Evidence, benign alternatives and fallback plan |

Suggested pass: 75/100; temporal correctness is mandatory.

## Verification and references
The authoring check runs this notebook sequentially on independent synthetic schema-compatible data, not Kaggle records. Actual IEEE-CIS metrics remain pending its downloaded training files. Default notebook outputs are cleared. The original CLI solutions remain supporting files, not separate required labs.

- [IEEE-CIS data dictionary](https://www.kaggle.com/competitions/ieee-fraud-detection/data)
- [Competition rules](https://www.kaggle.com/competitions/ieee-fraud-detection/rules)
- [DuckDB graph guide](https://duckdb.org/docs/current/guides/sql_features/graph_queries)
- [DuckPGQ property graph documentation](https://duckpgq.org/documentation/property_graph/)

PostgreSQL with Apache AGE is an alternative extension-based architecture, not implemented here. More advanced graph learning (GraphSAGE/GAT), community detection, production ingestion, real label-delay modelling and service deployment are beyond this session.
